In [2]:
import re

import networkx as nx
import pandas as pd
from rdflib import Graph, URIRef, Literal, Namespace, RDF, Variable
import warnings
import time
from rdflib.plugins.stores.sparqlstore import SPARQLStore
from sympy import print_tree, false
from collections import defaultdict

In [3]:
dataset_folder = 'datasets/'
dataset_name= 'NELL995'

rules_file= f'rule_mining/{dataset_name}/split_mined_rules-100'
train = f'{dataset_folder}/{dataset_name}/NELL995_train.tsv'
valid = f'{dataset_folder}/{dataset_name}/NELL995_valid.tsv'
test = f'{dataset_folder}/{dataset_name}/NELL995_test.tsv'

In [4]:
#load NELL ontology, recover functional properties
onto = Graph().parse('../datasets/NELL995/NELL.ontology.ttl')

query4functional = '''
prefix xsd:     <http://www.w3.org/2001/XMLSchema#>
prefix nellonto:  <http://ste-lod-crew.fr/nell/ontology/>
prefix owl:     <http://www.w3.org/2002/07/owl#>
prefix rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
prefix rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
select distinct ?prop where {
    ?prop a owl:FunctionalProperty .
}
'''
res4functional = onto.query(query4functional)

# Print the results
functional_properties = [row.prop for row in res4functional]

In [4]:
#sort and select top N rules

N=3000

def parse_rule(line : str):
    conf, rule = line.replace('<=', '').split('\t')[-2:]
    # pattern = re.compile(r'(<\w+>)\((\w+),(\w+)\)')
    # pattern = re.compile(r'(<[^>]+>)\s*\(([^,]+),([^)]+)\)')
    pattern = re.compile(r'(\w+)\((\w+),(\w+)\)')
    conf = float(conf)
    matches = pattern.findall(rule)
    return (conf,matches)

rules = []
with open(rules_file) as rf:
    for line in rf.readlines():
        rules.append(parse_rule(line))
rules = sorted(rules, key=lambda x: x[0], reverse=True)

#to materialize
top_N = [r[1] for r in rules[:N]]

#indexto to  hopefully make prediction faster
pred_rules_index = dict()
for rule in rules:
    if rule[1][0][0] not in pred_rules_index.keys():
        pred_rules_index[rule[1][0][0]] = [rule]
    else:
        pred_rules_index[rule[1][0][0]].append(rule)

# Link prediction - Custom solution

### Generate nx multidigraph

In [5]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    warnings.filterwarnings('ignore')

    newg=nx.MultiDiGraph()
    with open(train, 'r') as rf:
        for line in rf.readlines():
            s,p,o = line.strip('\n').split('\t')
            newg.add_edge(s,o, key=p)


### Domain-range

In [6]:
### builds a dict of properties which have domain and range defined

### this can only work because of the simplified structure of nell where dom/range are limited to 1 class
onto = Graph().parse('../datasets/NELL995/NELL.ontology.ttl')

query4domrange = '''
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT
  ?property
  (GROUP_CONCAT(DISTINCT STR(?domain); separator=" | ") AS ?domains)
  (GROUP_CONCAT(DISTINCT STR(?range); separator=" | ") AS ?ranges)
WHERE {
  ?property rdfs:domain|rdfs:range [] .
  OPTIONAL { ?property rdfs:domain ?domain . }
  OPTIONAL { ?property rdfs:range ?range . }
}
GROUP BY ?property

ORDER BY ?property
'''

query_disjoint = '''
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
SELECT ?className ?dw
WHERE {
?className owl:disjointWith ?dw
}
ORDER BY ?class ?dw
'''

res4domrange = onto.query(query4domrange)

domain_range_dict = dict()
for res in res4domrange:
    #prop_dict[res.property] = (URIRef(res.domains),URIRef(res.ranges))
    domain_range_dict[str(res.property).split('/')[-1]] = (str(res.domains).split('/')[-1],str(res.ranges).split('/')[-1])

domain_range_dict['Thing'] = ([],[]) #no constraints for generic classes


res4disjoint = onto.query(query_disjoint)

disjoint_dict = defaultdict(lambda: 'Thing')
for dw_res in res4disjoint:
    className = str(dw_res["className"]).split('/')[-1]
    dw_class = str(dw_res["dw"]).split('/')[-1]
    if className not in disjoint_dict.keys():
        disjoint_dict[className] = [dw_class]
    else:
        disjoint_dict[className].append(dw_class)


### Definition of recursive pathfinding function

In [19]:
def lookforpath(kg:nx.MultiDiGraph,target_var:str,remaining_rule:list, last_assigned_variable:str, open_vars:list, grounded_vars:dict, results_list:set, limit:int, functional:bool, disjoint_req:list):
    '''

    :param kg: the input graph
    :param target_var: target head variable according to the analyzed rule
    :param remaining_rule: remaining patterns in the CP rule
    :param last_assigned_variable: variable that was assigned in the previous pass, this is used to check the directionality of the current pattern
    :param open_vars: list of variables yet to be assigned
    :param grounded_vars: dictionary of grounded variables
    :param results_list: list of current acceptable assignments of target_var
    :param limit: limit to the additional assignments to be explored (limits search and won't be noticed in the hits@k measure)
    :param functional: boolean indicating whether functional rules are involved
    :param disjoint_req: boolean indicating disjoint types
    :return:
    '''
    if len(open_vars) ==0: # finished successfully
        to_add = grounded_vars[target_var]
        if  to_add not in results_list:
            results_list.add(to_add)
        if functional and len(results_list)>2: # functional exception condition
            return False
        if to_add.split('_')[0] in disjoint_req: # dom/range exception condition
            return False
        return True

    target_prop = remaining_rule[0][0] #the type of edge


    if last_assigned_variable == remaining_rule[0][1]: #last grounded variable is the subject of the next triple pattern
        out_edges = kg.out_edges(grounded_vars[last_assigned_variable], keys=True)
        if len(out_edges)==0:
            return True
        target_out_edges = [edge for edge in out_edges if edge[2] == target_prop]

        current_variable= remaining_rule[0][2] #trying to ground the object

        for oe in target_out_edges:
            if len(results_list)>limit:

                return True
            if oe[1] not in grounded_vars.values(): #no going back, and also not picking an entity already assigned
                if not lookforpath(kg=kg,target_var=target_var,
                           remaining_rule=remaining_rule[1:],
                           last_assigned_variable=current_variable,
                           open_vars =[v for v in open_vars if v != current_variable],
                           grounded_vars={**grounded_vars, current_variable:oe[1]},
                           results_list=results_list,
                            limit=limit, functional=functional,disjoint_req=disjoint_req):
                    return False


    else: #last grounded variable is the object of the next triple pattern
        in_edges = kg.in_edges(grounded_vars[last_assigned_variable], keys=True)
        if len(in_edges)==0:
            return True
        target_in_edges = [edge for edge in in_edges if edge[2] == target_prop]

        current_variable= remaining_rule[0][1] #trying to ground the subject
        for ie in target_in_edges:
            if len(results_list)>limit:

                return True
            if ie[0] not in grounded_vars.values():
                if not lookforpath(kg=kg,target_var=target_var,
                                   remaining_rule=remaining_rule[1:],
                                   last_assigned_variable=current_variable,
                                   open_vars= [v for v in open_vars if v != current_variable],
                                   grounded_vars={**grounded_vars, current_variable:ie[0]},
                                   results_list=results_list, limit=limit, functional=functional,disjoint_req=disjoint_req):
                    return False

    return True

In [15]:
def test_triple_metric(kg, known_entity:str,target_entity:str, candidate_rules:list, limit:int, mask_object:bool = True, functional:bool = False, disjoint_req:list = list() ):
    hits1 = 0
    hits5 = 0
    mrr = 0.0
    predictions = dict()
    unique_predictions = set()

    for conf,cand in candidate_rules:
        if len(unique_predictions) > limit:
            break
        all_valid_groundings = set()
        open_variables = list(set([t[1] for t in cand] + [t[2] for t in cand]))
        if mask_object:
            counts = lookforpath(kg=kg,
                    remaining_rule=cand[1:],
                    target_var=cand[0][2],
                    last_assigned_variable=cand[0][1],
                    open_vars=[v for v in open_variables if v!= cand[0][1]],
                    grounded_vars={cand[0][1]:known_entity},
                    results_list= all_valid_groundings,
                    limit = limit, functional=functional, disjoint_req=disjoint_req)
        else: #the subject is being masked
            counts = lookforpath(kg=kg,
                    remaining_rule=cand[1:][::-1],
                    target_var=cand[0][1],
                    last_assigned_variable=cand[0][2],
                    open_vars=[v for v in open_variables if v!= cand[0][2]],
                    grounded_vars={cand[0][2]:known_entity},
                    results_list= all_valid_groundings,
                    limit = limit, functional=functional, disjoint_req=disjoint_req)
        if counts and len(all_valid_groundings) >0 :
            #update the list of predictions and the list of unique predictions
            if conf in predictions.keys():
                predictions[conf] = predictions[conf].union(all_valid_groundings)
            else:
                predictions[conf] = set(all_valid_groundings)
            unique_predictions = unique_predictions.union(all_valid_groundings)

    #build the ranking
    sorted_predictions = [pred for key in sorted(predictions.keys(), reverse = True) for pred in predictions[key]]
    #aggregate according to 'max rank' criterion: only consider the highest conf rule for each predicted target
    unique_sorted_predictions = list(dict.fromkeys(sorted_predictions))
    if len(unique_predictions)>0:
        rank = (unique_sorted_predictions.index(target_entity)+1) if target_entity in unique_sorted_predictions[:100] else 101
        if rank ==1:
            hits1 = hits1 + 1
        if rank < 5:
            hits5 = hits5 + 1
        if rank <= 100:
            mrr = mrr + 1/rank
    return hits1, hits5, mrr

### Test time

In [20]:
######## start test #########

test_triples = pd.read_csv(test, sep= '\t', header = None, names = ['s','p','o'])
test_triples = test_triples[:1000]#
hits1 = 0
hits5 = 0
mrr = 0.0
limit = 100

#for debug
cnt=0
# for each test triple
for i, trip in test_triples.iterrows():
    p = trip.iloc[1]
    s = trip.iloc[0]
    o = trip.iloc[2]

    if URIRef('http://ste-lod-crew.fr/nell/ontology/') + p in functional_properties:
        functional = True
        cnt += 1
    else :
        functional = False
    if i % 5000 ==0 :
        print(i)
    #can add a first check for functionals here

    candidate_rules = pred_rules_index[p] if p in pred_rules_index.keys() else []
    #if functional and any(data == p for u,v,data in newg.out_edges(s,keys=True)):
    if functional: # this checks if a (s,p,?) exists, which would except the functional rule
        outgoing = [data for u,v,data in newg.out_edges(s,keys=True)]
        if p in outgoing:
            print('here')
            pass
    hits1o, hits5o, mrro = test_triple_metric(kg = newg, known_entity = s, target_entity = o,
                                              candidate_rules = candidate_rules, limit = limit,
                                              functional= functional, disjoint_req = disjoint_dict[domain_range_dict[p][1]])
    hits1s, hits5s, mrrs = test_triple_metric(kg = newg, known_entity = o, target_entity = s,
                                              candidate_rules = candidate_rules, limit = limit,
                                              mask_object=False, functional= functional, disjoint_req = disjoint_dict[domain_range_dict[p][0]])
    hits1=  hits1 + hits1o + hits1s
    hits5= hits5 + hits5o +hits5s
    mrr = mrr + mrro + mrrs

print (f'hits@1: {hits1/(2*len(test_triples))}, hits@5: {hits5/(2*len(test_triples))}, mrr: {mrr/(2*len(test_triples))}')
print(time.time() - start)
print(cnt)
start = time.time()


0


UnboundLocalError: local variable 'func_violations' referenced before assignment

# Materializing

In [ ]:
# Suppress the specific RDFLib warning about serializaition

import logging
rdflib_logger = logging.getLogger('rdflib')
rdflib_logger.setLevel(logging.ERROR)

#parse the graphs
# ns = Namespace('http://ste-lod-crew.fr/nell/ontology/')
ns = Namespace('')
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    warnings.filterwarnings('ignore')

    g=Graph()
    rule_materialized_g = Graph()
    nm_rule_materialized_g = Graph()

    with open(train, 'r') as rf:
        for line in rf.readlines():
            s,p,o = line.strip('\n').split('\t')
            g.add((ns[URIRef(s)], ns[URIRef(p)], ns[URIRef(o)]))
            rule_materialized_g.add((ns[URIRef(s)], ns[URIRef(p)], ns[URIRef(o)]))
            nm_rule_materialized_g.add((ns[URIRef(s)], ns[URIRef(p)], ns[URIRef(o)]))

g.serialize('datasets/NELL995/NELL995_train.ttl', format='turtle')
print(len(g))

In [ ]:
def create_query(rule, limit = 1000):
    vars = list(set([t[1] for t in rule]+ [t[2] for t in rule])) #lazy

    q = """
    prefix nell: <http://ste-lod-crew.fr/nell/ontology/>
    SELECT *
    WHERE {
    """
    for tp in rule[1:]:

        q += f'?{tp[1]} nell:{tp[0]} ?{tp[2]} . '
    q += ' filter (' + ' && '.join( [f'?{vars[i]} != ?{vars[i+1]}'for i in range(len(vars)-1)] ) + ')'
    q += ' filter not exists {{ ?{} nell:{} ?{} }}'.format(rule[0][1], rule[0][0], rule[0][2])
    q += f"""}} limit {limit}"""
    return q

def create_query_graphdb(rule, limit = 1000):
    vars = list(set([t[1] for t in rule]+ [t[2] for t in rule]))
    q = """
    prefix nell: <http://ste-lod-crew.fr/nell/ontology/>
    SELECT *
    WHERE {
    SERVICE <http://localhost:7200/repositories/NELL-train> {
    """
    for tp in rule[1:]:

        q += f'?{tp[1]} nell:{tp[0]} ?{tp[2]} . '
    q += ' filter (' + ' && '.join( [f'?{vars[i]} != ?{vars[i+1]}'for i in range(len(vars)-1)] ) + ')'
    q += ' filter not exists {{ ?{} nell:{} ?{} }}'.format(rule[0][1], rule[0][0], rule[0][2])
    q += f"""}} }} limit {limit}"""
    return q



def create_query_minus(rule):

    q = """
    prefix nell: <http://ste-lod-crew.fr/nell/ontology/>
    SELECT *
    WHERE { {
    """
    for tp in rule[1:]:

        q += f'?{tp[1]} nell:{tp[0]} ?{tp[2]} . '
    q += '}} minus {{ ?{} nell:{} [] }}'.format(rule[0][1], rule[0][0], rule[0][2])
    q += """}"""
    return q
def create_nmr_query(rule):
    q= """
    prefix nell: <http://ste-lod-crew.fr/nell/ontology/>
    SELECT *
    WHERE {
    """
    for tp in rule[1:]:

        q += f'?{tp[1]} nell:{tp[0]} ?{tp[2]} . '
    q += 'filter not exists {{ ?{} nell:{} [] }}'.format(rule[0][1], rule[0][0]) #no triples of type (?s,p,_)
    q += """}"""
    return q

def get_new_triples(graph,rule, nm=False):
    return_triples = []

    if nm:
        q = create_nmr_query(rule)
    else:
        q = create_query(rule)

    query_result = graph.query(q)

    for row in query_result:
        head_tp = rule[0]
        return_triples.append((row[head_tp[1]], 'http://ste-lod-crew.fr/nell/ontology/'+head_tp[0], row[head_tp[2]]))

    return return_triples


In [ ]:
#materialize the rules
new_triples = []
new_nm_triples = []
i=0
for rule in top_N:


    new_rule_triples = get_new_triples(g,rule)
    if URIRef('http://ste-lod-crew.fr/nell/ontology/'+rule[0][0]) in functional_properties:

        new_nm_rule_triples = get_new_triples(g,rule,nm=True)
    else:
        new_nm_rule_triples = new_rule_triples

    new_triples += new_rule_triples
    new_nm_triples += new_nm_rule_triples

print(len(new_triples))
new_unique_triples = list(set(new_triples))
new_unique_nm_triples = list(set(new_nm_triples))
print(len(new_unique_triples))
print(len(new_unique_nm_triples))

In [ ]:
for trip in new_unique_triples:
    rule_materialized_g.add((URIRef(trip[0]), URIRef(trip[1]), URIRef(trip[2])))
for trip in new_unique_nm_triples:
    nm_rule_materialized_g.add((URIRef(trip[0]), URIRef(trip[1]), URIRef(trip[2])))
new_triples = []
new_nm_triples =[]
new_unique_triples = []
new_unique_nm_triples = []

# CONSISTENCY ANALYSES

In [ ]:
#looks for and prints properties that do ot satisfy functionality

def test_functionality(property):
    q= f'''
    select distinct ?subject ?object1 ?object2
    where {{

      ?subject <{property}> ?object1 .
      ?subject <{property}> ?object2 .
      FILTER (str(?object1) < str(?object2))
    }}
    '''
    return q

print('Original graph:')
inconsistent_triples_g = 0
for prop in functional_properties:
    answer= g.query(test_functionality(prop))
    inconsistent_triples_g += len(answer)
print(inconsistent_triples_g)


inconsistent_triples_rule_materialized_g =0
inconsistent_triples_nm_rule_materialized_g =0


for prop in functional_properties:
    answer= rule_materialized_g.query(test_functionality(prop))
    inconsistent_triples_rule_materialized_g += len(answer)

    answer= nm_rule_materialized_g.query(test_functionality(prop))
    inconsistent_triples_nm_rule_materialized_g += len(answer)

print('After materialization - normal rules:')
print(inconsistent_triples_rule_materialized_g)
print('After materialization - nm rules:')
print(inconsistent_triples_nm_rule_materialized_g)




# Link Prediction

In [ ]:
endpoint = "http://localhost:7200/repositories/NELL-train"

# Set up the SPARQLStore
store = SPARQLStore(endpoint)

# Create an RDFlib Graph object with the SPARQLStore
# This graph object doesn't hold any data itself; it's a gateway to GraphDB.
graphdbgraph = Graph(store)

In [ ]:
def subject_nmr_test_query_minus(rule, s, limit=100):
    vars = list(set([t[1] for t in rule] + [t[2] for t in rule]))  #lazy

    q = f"""
    prefix nell: <http://ste-lod-crew.fr/nell/ontology/>
    SELECT ?{rule[0][2]}
    WHERE {{

    """
    for tp in rule[1:]:
        q += f'?{tp[1]} nell:{tp[0]} ?{tp[2]} . '
    q += ' filter (' + ' && '.join([f'?{vars[i]} != ?{vars[i + 1]}' for i in range(len(vars) - 1)]) + ')'
    #q += 'filter not exists {{ ?{} nell:{} [] }}'.format(rule[0][1], rule[0][0])  #no triples of type (?s,p,_)
    q += f""" MINUS {{ ?{rule[0][1]} nell:{rule[0][0]} [] }} }} limit {limit}"""
    #q += f"""}} limit {limit}"""
    q = q.replace(f'?{rule[0][1]}', f'nell:{s}')
    return q


def find_rank(aggregated, target):
    for index, (string, _) in enumerate(aggregated):
        if string == target:
            return index
    return float('inf')

def compute_rank_metrics(train_graph,test_set,limit=100, debug= -1):
    # read test set


    test_triples = pd.read_csv(test_set, sep= '\t', header = None, names = ['s','p','o'])
    if debug > 0:
        test_triples = test_triples[:debug]
    last_good_q = '' #for debug
    hits1 = 0
    hits5 = 0


    # for each test triple
    start = time.time()
    for i, trip in test_triples.iterrows():
        if (i % 100) ==0 :
            print(i)

        #check all rules that can predict it
        if trip.p in pred_rules_index.keys():
            if trip.p in functional_properties:
                updatedlimit= limit
            else:
                updatedlimit= 1
            candidate_rules = pred_rules_index[trip.p]
            candidate_objects = dict()


            for c in candidate_rules:

                conf,candidate_r = c
                target = Variable(candidate_r[0][2])

                test_query = subject_nmr_test_query_minus(candidate_r, trip.s, limit=updatedlimit)
                try:
                    predicted_triples= train_graph.query(test_query)

                except Exception as e:
                    print(e)
                    print(test_query)
                    return -1,-1
                if len(predicted_triples)>0:
                    last_good_q = test_query
                    for binding in predicted_triples.bindings:

                        #new_object  =binding[f'{target}'].split('/')[-1]
                        new_object  =binding[target].split('/')[-1]
                        if new_object not in candidate_objects.keys():
                            candidate_objects[new_object] = [conf]
                        else:
                            candidate_objects[new_object].append(conf)

                    #print(stop)
                #we only care about top limit candidates

                if len(candidate_objects.keys()) > limit:
                    #print('beyond limit')
                    break
            aggregated_objects=  sorted(candidate_objects.items(), key=lambda x: max(x[1]), reverse=True)
            rank = find_rank(aggregated_objects, trip.o)
            if rank <1:
                hits1 = hits1 + 1
            if rank < 5:
                hits5 = hits5 + 1
    stop = time.time() - start
    print(stop)
    print(last_good_q)
    return hits1/len(test_triples), hits5/len(test_triples)


print('graphdb')
h1,h5= compute_rank_metrics(graphdbgraph,test, limit = 1,debug = 1)
print(h5)

#partially materialize the bod
    #or confidence,candidate in candidates:

#query and rank

#compute ranking

### Rest

In [ ]:
### checks if there are domain inconsistencies when adding new triples
from itertools import combinations
# def domain_range_check():
#     return ''' PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
#             select *
#             #?subject ?prop1 ?object1 ?prop2 ?object2
#             where {
#              #?subject ?prop1 ?object1 .
#
#              ?subject ?prop2 ?object2 .
#              filter(?prop1 != ?prop2) .
#              ?prop1 rdfs:domain ?domain1 .
#              ?prop2 rdfs:domain ?domain2 .
#              filter (?domain1 != ?domain2) .
#             }
#             '''


#this is slow, checks for subclassing
def domain_comparison_q(d1,d2):
    q= '''
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        ASK {
          # Bind your specific classes here for a concrete check
          # BIND(<%s> AS ?d1)
          # BIND(<%s> AS ?d2)

          { ?d1 rdfs:subClassOf* ?d2 . }
          UNION
          { ?d2 rdfs:subClassOf* ?d1 . }
        }

        '''%(d1,d2)
    return q

#this is faster, checks for disjointedness, but requires explicit reasoning
def disjoint_check_q(d1,d2):
    q= '''
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        ASK {
          BIND(<%s> AS ?d1)
          BIND(<%s> AS ?d2)

          ?d1 owl:disjointWith ?d2 .
        }

        '''%(d1,d2)
    return q



# rule_materialized_g_and_onto = rule_materialized_g + onto
# qdr_res = rule_materialized_g_and_onto.query(qdr)

In [ ]:
subjects_to_check = {s for s, p, o in new_triples}
inconsistencies = set()

for s in subjects_to_check:
    props_for_subject = {
        p for p, o in rule_materialized_g.predicate_objects(s)
    }
    if len(props_for_subject) < 2:
            continue
    for prop1, prop2 in combinations(props_for_subject, 2):
            domain1 = prop_dict.get(prop1)[0]
            domain2 = prop_dict.get(prop2)[0]
            if domain1 and domain2 and domain1 != domain2:
                # Sort to ensure (p1, p2) is treated the same as (p2, p1)
                sorted_props = tuple(sorted((prop1, prop2), key=str))
                sorted_domains = tuple(
                    (domain1, domain2) if prop1 == sorted_props[0] else (domain2, domain1)
                )

                inconsistency_record = (s, sorted_props, sorted_domains)
                inconsistencies.add(inconsistency_record)

remaining = []
for inc in inconsistencies:
    d1 = inc[2][0]
    d2 = inc[2][1]
    #query_result = onto.query(domain_comparison_q(d1,d2))
    query_result = onto.query(disjoint_check_q(d1,d2))
    if query_result.askAnswer:
        remaining.append(inc)

In [ ]:
#validate both graphs

from pyshacl import validate


base_conforms, base_results_graph, base_results_text = validate(data_graph=g,
                                                 shacl_graph=f'Yago4/yago-wd-shapes.nt',
                                                 ont_graph=f'Yago4/yago-wd-class.nt',
                                                                max_validation_depth=32)


#
# extended_conforms, extended_results_graph, extended_results_text = validate(data_graph=rule_materialized_g,
#                                                                             shacl_graph=f'Yago4/yago-wd-shapes.nt',
#                                                                             ont_graph=f'Yago4/yago-wd-schema.nt')

In [ ]:
query_shacl = """
    PREFIX sh: <http://www.w3.org/ns/shacl#>
    SELECT distinct ?violation WHERE {
        ?s sh:resultMessage ?violation.
    }
"""

qres_base = base_results_graph.query(query_shacl)

# Print the results
for row in qres_base:
    print(f"{row.violation}")

print(len(base_results_graph))


In [ ]:

new_conforms, new_results_graph, new_results_text = validate(data_graph=new_graph,
                                                                            shacl_graph=f'Yago4.5/yago-schema.ttl', ont_graph=f'Yago4.5/yago-taxonomy.ttl', debug=True)
qres_new = new_results_graph.query(query_shacl)



# Print the results
for row in qres_new:
    print(f"{row.violation}")

print(len(new_results_graph))

In [ ]:
print(len(g))
print(len(rule_materialized_g))

In [ ]:
qres_extended = extended_results_graph.query(query_shacl)

# Print the results
for row in qres_extended:
    print(f"{row.violation}")

print(len(extended_results_graph))



In [ ]:
info= """
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX yago: <http://yago-knowledge.org/resource/>
PREFIX schema: <http://schema.org/>

SELECT distinct * WHERE {
    ?child schema:parent ?parent .
    ?parent schema:gender ?g .
    filter not exists {{ ?child schema:gender ?g .}}
}
"""

info_res = g.query(info)
for row in info_res:
    print(row.child + '\t' + row.g)


In [ ]:
# #for yago
# shapes = Graph().parse('datasets/Yago4/yago-wd-shapes.nt')
#
# query4functional = '''
# select distinct ?prop where {
#     ?shape <http://www.w3.org/ns/shacl#maxCount> ?limit .
#     ?shape <http://www.w3.org/ns/shacl#path> ?prop .
#     filter (?limit = 1)
# }
# '''
# res4functional = shapes.query(query4functional)
#
# # Print the results
# functional_properties = [row.prop for row in res4functional]

In [ ]:
#add new triples to the graph
# new_graph = Graph()
#
# with open(f'rule_mining/{dataset_name}/new_triples.txt', 'w') as wf:
#     for t in  new_unique_triples:
#         new_graph.add((t[0], URIRef(t[1]), t[2]))
#         wf.write(f'{str(t[0])}\t{t[1]}\t{str(t[2])}\n')
# print(len(new_graph))
#
# #also add taxonomy and schema info
#
# # with open(f'{dataset_folder}/{dataset_name}/ent2classes.txt', 'r') as rf:
# #     for l in rf.readlines():
# #         s,o = l.split('\t')
# #         new_graph.add((URIRef(s), RDF.type, URIRef(o.strip('\n'))))
# #         g.add((URIRef(s), RDF.type, URIRef(o.strip('\n'))))
#
#
# rule_materialized_g = rule_materialized_g + new_graph
# nm_rule_materialized_g = nm_rule_materialized_g + new_graph
# print(len(rule_materialized_g))